# Phase 2 Reasoning on Mixed Emotion Dataset

This notebook runs the paper-aligned Phase 2 reasoning prompts on the supplementary Mixed Emotion Dataset.

- Llama 2 + Chain-of-Thought prompting -> `LLaMA2_1`, `LLaMA2_2`, `LLaMA2_3`, `LLaMA2_final_label`
- Llama 3 + SELF-DISCOVER prompting -> `LLaMA3_SELECT`, `LLaMA3_ADAPT`, `LLaMA3_IMPLEMENT`, `LLaMA3_Answer`, `LLaMA3_final_label`

The prompt wording is aligned with Appendix Table A2 and Appendix Table A3 in the current manuscript.


In [ ]:
# Colab setup. Run this cell first in a fresh Colab runtime.
!pip -q install -U pandas tqdm scikit-learn transformers accelerate bitsandbytes sentencepiece protobuf


In [ ]:
import os
import gc
import re
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
except Exception:
    accuracy_score = classification_report = confusion_matrix = None

LABELS = ["Depression", "Neutral", "Happy"]


## Configuration

For a quick smoke test, keep `MAX_ROWS = 3`. For the full 300-example dataset, set `MAX_ROWS = None`.

If there is no Phase 1 prediction file yet, `PHASE1_LABEL_MODE = "target_as_placeholder"` uses the reference label as the AI-generated label placeholder in the prompt. Later, when Phase 1 outputs are available, set `PHASE1_LABEL_MODE = "prediction_column"` and provide a `prediction` column.


In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/WoojinPark-Jay/"
    "confidence-guided-llm-reasoning-depression-risk-emotion/"
    "refs/heads/main/data/supplementary/mixed_emotion/"
    "mixed_emotion_stress_test_v2_2_300.csv"
)

# Optional local path for local notebook testing. In Colab this path will usually not exist.
LOCAL_DATA_PATH = "/Users/woojinpark/LocalDocuments/헬스케어 논문/confidence-guided-selective-llm-reasoning/data/supplementary/mixed_emotion/mixed_emotion_stress_test_v2_2_300.csv"

OUTPUT_DIR = Path("outputs_phase2_reasoning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COL = "text"
TRUE_LABEL_COL = "target_label"
PREDICTION_COL = "prediction"

# "target_as_placeholder": use target_label as temporary AI-generated label when Phase 1 is absent.
# "prediction_column": use an existing Phase 1 prediction column.
PHASE1_LABEL_MODE = "target_as_placeholder"

MAX_ROWS = 300  # use all 300 mixed-emotion examples; set to a small number such as 3 only for smoke testing
RANDOM_STATE = 42

RUN_LLAMA2_COT = True
RUN_LLAMA3_SELF_DISCOVER = True

LLAMA2_MODEL_NAME = "NousResearch/Llama-2-7b-chat-hf"
LLAMA3_MODEL_NAME = "NousResearch/Meta-Llama-3-8B-Instruct"

# SELF-DISCOVER mode:
# - "per_sample": generate SELECT/ADAPT/IMPLEMENT for each row, closest to the original code but slow.
# - "fixed": use one reusable paper-safe reasoning structure, faster for smoke tests.
SELF_DISCOVER_STRUCTURE_MODE = "per_sample"

MAX_NEW_TOKENS_COT = 256
MAX_NEW_TOKENS_SELF_DISCOVER = 1024


In [ ]:
# Optional Hugging Face token support for gated or rate-limited model access.
# In Colab, add a secret named HF_TOKEN if needed.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab secrets.")
except Exception:
    print("Colab userdata not available or HF_TOKEN not set. Continuing without explicit HF token.")


## Load Mixed Emotion Dataset

In [ ]:
def load_mixed_emotion_dataset():
    local_path = Path(LOCAL_DATA_PATH)
    if local_path.exists():
        print(f"Loading local dataset: {local_path}")
        return pd.read_csv(local_path)
    print(f"Loading dataset from GitHub raw URL:\n{DATA_URL}")
    return pd.read_csv(DATA_URL)

df = load_mixed_emotion_dataset()
print(df.shape)
display(df.head())
print(df[TRUE_LABEL_COL].value_counts())


In [ ]:
required = {TEXT_COL, TRUE_LABEL_COL}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

work_df = df.copy()
if PHASE1_LABEL_MODE == "target_as_placeholder":
    work_df["phase1_label_for_prompt"] = work_df[TRUE_LABEL_COL]
elif PHASE1_LABEL_MODE == "prediction_column":
    if PREDICTION_COL not in work_df.columns:
        raise ValueError(f"PHASE1_LABEL_MODE='prediction_column' but '{PREDICTION_COL}' is missing.")
    work_df["phase1_label_for_prompt"] = work_df[PREDICTION_COL]
else:
    raise ValueError("Unknown PHASE1_LABEL_MODE")

if MAX_ROWS is not None:
    work_df = work_df.sample(n=min(MAX_ROWS, len(work_df)), random_state=RANDOM_STATE).reset_index(drop=True)
else:
    work_df = work_df.reset_index(drop=True)

print(work_df.shape)
display(work_df[["example_id", TRUE_LABEL_COL, "phase1_label_for_prompt", TEXT_COL]].head())


## Model Loading Helpers

In [ ]:
def load_chat_model(model_name, load_in_4bit=True):
    compute_dtype = torch.float16
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    quantization_config = None
    if load_in_4bit:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=compute_dtype,
        device_map="auto",
        quantization_config=quantization_config,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    )
    model.eval()
    return tokenizer, model

def clear_model(tokenizer=None, model=None):
    del tokenizer
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Appendix B-Aligned Llama 2 Chain-of-Thought Prompt

In [ ]:
LLAMA2_COT_REQUESTS = [
    "You are an expert annotator for mental-health-related emotion classification. Assist in analyzing emotions in text data. This task is intended for text-based emotion classification, not clinical diagnosis.",
    """I will provide you with a piece of text along with an AI-generated emotional classification label.
The text may contain one or more of the following emotions: Depression, Neutral, and Happy.
Your task is to assess the emotional tone of the text and determine whether the AI's classification is accurate.
Do not make a clinical diagnosis, infer a medical condition, or provide treatment advice.""",
    """Objectively analyze the emotions expressed in the text and identify the dominant emotion that best represents the overall sentiment.
If the text conveys persistent sadness, hopelessness, or emotional distress as the dominant tone, classify it as Depression.
If the text lacks strong emotions and appears balanced, factual, or emotionally neutral, classify it as Neutral.
If the text conveys happiness, accomplishment, relief, or fulfillment as the dominant sentiment, classify it as Happy.""",
    "Compare your emotional analysis with the AI's predicted label. Do they match? If not, determine the correct classification based on the dominant emotional tone of the text.",
    """Finally, provide the percentage breakdown of emotions in the text in the order of Depression, Neutral, Happy.
Your response should consist of only three numerical values separated by commas, without percent signs and without any additional explanation.
The largest value is used downstream as the final Phase 2 label.""",
]

def format_llama2_input(text, phase1_label):
    return f"Text:\n{text}\n\nAI-generated emotional classification label:\n{phase1_label}"


In [ ]:
def chat_generate(tokenizer, model, messages, max_new_tokens=256, do_sample=False, temperature=None, top_p=None):
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    terminators = [tokenizer.eos_token_id]
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot_id, int) and eot_id >= 0:
        terminators.append(eot_id)

    kwargs = dict(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=do_sample,
    )
    if temperature is not None:
        kwargs["temperature"] = temperature
    if top_p is not None:
        kwargs["top_p"] = top_p

    with torch.no_grad():
        outputs = model.generate(**kwargs)
    response = outputs[0][input_ids.shape[-1]:]
    return tokenizer.decode(response, skip_special_tokens=True).strip()

def run_llama2_cot_one(tokenizer, model, text, phase1_label):
    messages = [
        {"role": "system", "content": LLAMA2_COT_REQUESTS[0]},
        {"role": "user", "content": LLAMA2_COT_REQUESTS[1]},
    ]
    messages.append({"role": "assistant", "content": chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)})
    messages.append({"role": "user", "content": format_llama2_input(text, phase1_label)})
    messages.append({"role": "assistant", "content": chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)})

    answers = []
    for question in LLAMA2_COT_REQUESTS[2:]:
        messages.append({"role": "user", "content": question})
        answer = chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)
        messages.append({"role": "assistant", "content": answer})
        answers.append(answer)
    return answers

def parse_llama2_final_label(output):
    numbers = re.findall(r"[-+]?\d*\.?\d+", str(output))
    if len(numbers) < 3:
        return np.nan
    scores = [float(x) for x in numbers[:3]]
    return LABELS[int(np.argmax(scores))]


In [ ]:
llama2_results = None

if RUN_LLAMA2_COT:
    tokenizer, model = load_chat_model(LLAMA2_MODEL_NAME, load_in_4bit=True)
    rows = []
    for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc="Llama 2 CoT"):
        answers = run_llama2_cot_one(
            tokenizer,
            model,
            row[TEXT_COL],
            row["phase1_label_for_prompt"],
        )
        rows.append({
            "example_id": row.get("example_id", None),
            "text": row[TEXT_COL],
            "target_label": row[TRUE_LABEL_COL],
            "phase1_label_for_prompt": row["phase1_label_for_prompt"],
            "LLaMA2_1": answers[0] if len(answers) > 0 else np.nan,
            "LLaMA2_2": answers[1] if len(answers) > 1 else np.nan,
            "LLaMA2_3": answers[2] if len(answers) > 2 else np.nan,
        })
    llama2_results = pd.DataFrame(rows)
    llama2_results["LLaMA2_final_label"] = llama2_results["LLaMA2_3"].apply(parse_llama2_final_label)
    llama2_path = OUTPUT_DIR / "mixed_emotion_llama2_cot_results.csv"
    llama2_results.to_csv(llama2_path, index=False)
    print(f"Saved: {llama2_path}")
    display(llama2_results.head())
    clear_model(tokenizer, model)


## Appendix C-Aligned Llama 3 SELF-DISCOVER Prompt

The SELF-DISCOVER select/adapt/implement prompt templates are embedded here from `prompts.py`, so this notebook does not need a separate `prompts.py` upload in Colab.


In [ ]:
reasoning_modules = """
1 How could I devise an experiment to help solve that problem?
2 Make a list of ideas for solving this problem, and apply them one by one to the problem to see if any progress can be made.
3 How could I measure progress on this problem?
4 How can I simplify the problem so that it is easier to solve?
5 What are the key assumptions underlying this problem?
6 What are the potential risks and drawbacks of each solution?
7 What are the alternative perspectives or viewpoints on this problem?
8 What are the long-term implications of this problem and its solutions?
9 How can I break down this problem into smaller, more manageable parts?
10 Critical Thinking: This style involves analyzing the problem from different perspectives, questioning assumptions, and evaluating
the evidence or information available. It focuses on logical reasoning, evidence-based decision-making, and identifying
potential biases or flaws in thinking.
11 Try creative thinking, generate innovative and out-of-the-box ideas to solve the problem. Explore unconventional solutions,
thinking beyond traditional boundaries, and encouraging imagination and originality.
12 Seek input and collaboration from others to solve the problem. Emphasize teamwork, open communication, and leveraging the
diverse perspectives and expertise of a group to come up with effective solutions.
13 Use systems thinking: Consider the problem as part of a larger system and understanding the interconnectedness of various elements.
Focuses on identifying the underlying causes, feedback loops, and interdependencies that influence the problem, and developing holistic
solutions that address the system as a whole.
14 Use Risk Analysis: Evaluate potential risks, uncertainties, and tradeoffs associated with different solutions or approaches to a
problem. Emphasize assessing the potential consequences and likelihood of success or failure, and making informed decisions based
on a balanced analysis of risks and benefits.
15 Use Reflective Thinking: Step back from the problem, take the time for introspection and self-reflection. Examine personal biases,
assumptions, and mental models that may influence problem-solving, and being open to learning from past experiences to improve
future approaches.
16 What is the core issue or problem that needs to be addressed?
17 What are the underlying causes or factors contributing to the problem?
18 Are there any potential solutions or strategies that have been tried before? If yes, what were the outcomes and lessons learned?
19 What are the potential obstacles or challenges that might arise in solving this problem?
20 Are there any relevant data or information that can provide insights into the problem? If yes, what data sources are available,
and how can they be analyzed?
21 Are there any stakeholders or individuals who are directly affected by the problem? What are their perspectives and needs?
22 What resources (financial, human, technological, etc.) are needed to tackle the problem effectively?
23 How can progress or success in solving the problem be measured or evaluated?
24 What indicators or metrics can be used?
25 Is the problem a technical or practical one that requires a specific expertise or skill set? Or is it more of a conceptual or
theoretical problem?
26 Does the problem involve a physical constraint, such as limited resources, infrastructure, or space?
27 Is the problem related to human behavior, such as a social, cultural, or psychological issue?
28 Does the problem involve decision-making or planning, where choices need to be made under uncertainty or with competing
objectives?
29 Is the problem an analytical one that requires data analysis, modeling, or optimization techniques?
30 Is the problem a design challenge that requires creative solutions and innovation?
31 Does the problem require addressing systemic or structural issues rather than just individual instances?
32 Is the problem time-sensitive or urgent, requiring immediate attention and action?
33 What kinds of solution typically are produced for this kind of problem specification?
34 Given the problem specification and the current best solution, have a guess about other possible solutions.
35 Let's imagine the current best solution is totally wrong, what other ways are there to think about the problem specification?
36 What is the best way to modify this current best solution, given what you know about these kinds of problem specification?
37 Ignoring the current best solution, create an entirely new solution to the problem.
38 Let's think step by step.
39 Let's make a step by step plan and implement it with good notion and explanation"""


select_prompt = """
In order to solve the given task:
<Task>
{Task}
</Task>
Select several modules that are crucial for solving the tasks above
from all the reasoning module description given below:
{resonining_modules}
"""

adapt_prompt = """
Rephrase and specify each reasoning module so that it better helps solving the task:
<Task>
{Task}
</Task>
SELECTED module descriptions:
{selected_modules}
Adapt each reasoning module description to better solve the task:
"""

implement_prompt = """
Operationalize the reasoning modules into a step-by-step reasoning plan in JSON format
Example task:
<Task>
{Task}
</Task>
ADAPTED module descriptions:
{adapted_modules}

Implement a reasoning structure to generalise similar task to follow step-by-step and arrive at correct answers
"""

In [ ]:
SELF_DISCOVER_TASK_TEMPLATE = """
You are an expert annotator for mental-health-related emotion classification.
Your task is to analyze the emotional content of text data using structured reasoning.
This task is intended for research-oriented text classification, not clinical diagnosis.

I will provide you with text data and an AI-generated emotional classification label.
Your task is to determine the dominant emotion that best represents the overall sentiment of the text.
The text may contain multiple emotions, but your goal is to determine the most representative emotion that captures the overall tone.
Do not make a clinical diagnosis, infer a medical condition, or provide treatment advice.

<context>

data: {data}

result: {label}

<questions>

1. Objectively analyze the given text. Identify the dominant emotion that best represents the text's overall sentiment.
2. Compare your analysis from Question 1 with the AI's classified label.
3. Evaluate the AI's classification using textual evidence. If it aligns with your independent assessment, confirm it. If it does not, determine the correct dominant emotion and justify the decision using only evidence from the text.
4. Select the option that most accurately represents your analysis. Do not create or explain additional choices beyond the provided options. End the response with exactly one final label using the format Final label: [label].
<Options>Depression, Neutral, Happy</Options>

Classification Guidelines:
- If the text expresses ongoing sadness, hopelessness, emotional distress, or a strongly negative emotional trajectory, classify it as Depression.
- If the text is mainly factual, balanced, informational, or emotionally mild, classify it as Neutral.
- If the text expresses happiness, accomplishment, relief, gratitude, or fulfillment as the dominant sentiment, classify it as Happy.
- When multiple emotions are present, choose the class that best reflects the overall message rather than isolated phrases.

Output Constraint:
Do not create labels outside the provided options.
When explaining the decision, base the justification on textual evidence rather than clinical assumptions.
The response should end with exactly one final label in the format Final label: Depression, Final label: Neutral, or Final label: Happy.

</questions>
"""

FIXED_SELF_DISCOVER_STRUCTURE = """
Use a structured, text-grounded reasoning plan:
1. Identify the dominant emotional cues in the text.
2. Compare the dominant emotional trajectory with the AI-generated label.
3. Check whether mixed or shifting cues change the most representative emotion.
4. Justify the decision using textual evidence only.
5. End with exactly one label in the format Final label: Depression, Final label: Neutral, or Final label: Happy.
"""

def build_self_discover_task(text, phase1_label):
    return SELF_DISCOVER_TASK_TEMPLATE.replace("{data}", str(text)).replace("{label}", str(phase1_label))

def parse_llama3_final_label(output):
    text = str(output)
    match = re.search(r"Final label\s*:\s*(Depression|Neutral|Happy)", text, flags=re.IGNORECASE)
    if match:
        value = match.group(1).lower()
        return next(label for label in LABELS if label.lower() == value)
    for label in LABELS:
        if re.search(rf"\b{label}\b", text, flags=re.IGNORECASE):
            return label
    return np.nan


In [ ]:
def run_self_discover_structure(tokenizer, model, task):
    select = select_prompt.replace("{Task}", task).replace("{resonining_modules}", reasoning_modules)
    selected_modules = chat_generate(tokenizer, model, [{"role": "user", "content": select}], MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=True, top_p=1.0)

    adapt = adapt_prompt.replace("{Task}", task).replace("{selected_modules}", selected_modules)
    adapted_modules = chat_generate(tokenizer, model, [{"role": "user", "content": adapt}], MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=True, top_p=1.0)

    implement = implement_prompt.replace("{Task}", task).replace("{adapted_modules}", adapted_modules)
    reasoning_structure = chat_generate(tokenizer, model, [{"role": "user", "content": implement}], MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=True, top_p=1.0)
    return selected_modules, adapted_modules, reasoning_structure

def run_llama3_self_discover_one(tokenizer, model, text, phase1_label):
    task = build_self_discover_task(text, phase1_label)
    if SELF_DISCOVER_STRUCTURE_MODE == "per_sample":
        selected, adapted, structure = run_self_discover_structure(tokenizer, model, task)
    elif SELF_DISCOVER_STRUCTURE_MODE == "fixed":
        selected = "Fixed paper-safe reasoning modules: textual evidence review, label comparison, ambiguity handling, and final constrained label selection."
        adapted = "Adapted to identify dominant emotional trajectory, compare with the AI-generated label, and avoid clinical inference."
        structure = FIXED_SELF_DISCOVER_STRUCTURE
    else:
        raise ValueError("SELF_DISCOVER_STRUCTURE_MODE must be 'per_sample' or 'fixed'.")

    final_prompt = (
        f"Using the following reasoning structure:\n{structure}\n\n"
        f"Solve this task, providing your answer:\n{task}\n\n"
        "Note1: Write the question number before your answer.\n"
        "Note2: Do not write anything besides your answer.\n"
        "Note3: End with exactly one final label in the format Final label: Depression, Final label: Neutral, or Final label: Happy."
    )
    messages = [
        {"role": "system", "content": "You are an expert annotator for research-oriented, non-clinical emotion classification. Do not provide clinical diagnosis, medical inference, treatment advice, or professional mental health advice."},
        {"role": "user", "content": final_prompt},
    ]
    answer = chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=False)
    return selected, adapted, structure, answer


In [ ]:
llama3_results = None

if RUN_LLAMA3_SELF_DISCOVER:
    tokenizer, model = load_chat_model(LLAMA3_MODEL_NAME, load_in_4bit=True)
    rows = []
    for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc="Llama 3 SELF-DISCOVER"):
        selected, adapted, structure, answer = run_llama3_self_discover_one(
            tokenizer,
            model,
            row[TEXT_COL],
            row["phase1_label_for_prompt"],
        )
        rows.append({
            "example_id": row.get("example_id", None),
            "text": row[TEXT_COL],
            "target_label": row[TRUE_LABEL_COL],
            "phase1_label_for_prompt": row["phase1_label_for_prompt"],
            "LLaMA3_SELECT": selected,
            "LLaMA3_ADAPT": adapted,
            "LLaMA3_IMPLEMENT": structure,
            "LLaMA3_Answer": answer,
        })
    llama3_results = pd.DataFrame(rows)
    llama3_results["LLaMA3_final_label"] = llama3_results["LLaMA3_Answer"].apply(parse_llama3_final_label)
    llama3_path = OUTPUT_DIR / "mixed_emotion_llama3_self_discover_results.csv"
    llama3_results.to_csv(llama3_path, index=False)
    print(f"Saved: {llama3_path}")
    display(llama3_results.head())
    clear_model(tokenizer, model)


## Evaluation

This section evaluates Phase 2 final labels against `target_label`. When Phase 1 predictions are added later, this can be expanded to correction counts, introduced errors, and net corrections.


In [ ]:
def evaluate_final_labels(result_df, pred_col, name):
    if result_df is None or pred_col not in result_df.columns:
        print(f"{name}: no results to evaluate.")
        return None
    eval_df = result_df.dropna(subset=[pred_col]).copy()
    if eval_df.empty:
        print(f"{name}: no parseable final labels.")
        return None
    acc = (eval_df[pred_col] == eval_df["target_label"]).mean()
    print(f"\n{name}")
    print(f"Rows evaluated: {len(eval_df)} / {len(result_df)}")
    print(f"Accuracy vs target_label: {acc:.4f}")
    print(pd.crosstab(eval_df["target_label"], eval_df[pred_col], rownames=["target"], colnames=[pred_col]))
    if classification_report is not None:
        print(classification_report(eval_df["target_label"], eval_df[pred_col], labels=LABELS, zero_division=0))
    return {"model": name, "rows": len(eval_df), "accuracy": acc}

summary = []
item = evaluate_final_labels(llama2_results, "LLaMA2_final_label", "Llama 2 CoT")
if item: summary.append(item)
item = evaluate_final_labels(llama3_results, "LLaMA3_final_label", "Llama 3 SELF-DISCOVER")
if item: summary.append(item)

summary_df = pd.DataFrame(summary)
if not summary_df.empty:
    summary_path = OUTPUT_DIR / "mixed_emotion_phase2_reasoning_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Saved: {summary_path}")
    display(summary_df)


## Combined Output Table

In [ ]:
combined = work_df[["example_id", TEXT_COL, TRUE_LABEL_COL, "phase1_label_for_prompt"]].copy()
if llama2_results is not None:
    combined = combined.merge(
        llama2_results[["example_id", "LLaMA2_1", "LLaMA2_2", "LLaMA2_3", "LLaMA2_final_label"]],
        on="example_id",
        how="left",
    )
if llama3_results is not None:
    combined = combined.merge(
        llama3_results[["example_id", "LLaMA3_SELECT", "LLaMA3_ADAPT", "LLaMA3_IMPLEMENT", "LLaMA3_Answer", "LLaMA3_final_label"]],
        on="example_id",
        how="left",
    )
combined_path = OUTPUT_DIR / "mixed_emotion_phase2_reasoning_combined_outputs.csv"
combined.to_csv(combined_path, index=False)
print(f"Saved: {combined_path}")
display(combined.head())
